# SIM V2 3D — Conformal Phase B + C (11-channel effective medium)

Generate **11-channel** training data from the **conformal effective-medium** 7th floor — continuous
per-cell εr/σ (subcell-averaged from the real CAD polygons, 253 materials → ITU P.2040), not the old
hard 6-class grid — then train the pure-JAX 3-D U-Net (cin=11).

Ships the **baked whole-floor εr/σ grids** (`SIM V1 3D/conformal/LTE_B71_617/`), so it runs **without
the 344 MB OBJ**: each Tx crops + zooms those grids to the FDTD resolution and solves the
continuous-medium field (`EffectiveMediumScene`). **Pick a GPU runtime** (Runtime → Change runtime
type → GPU) for scale.

In [1]:
# === Environment bootstrap: Colab or local ===
import sys
from pathlib import Path
try:
    import google.colab            # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    BRANCH = "sim-v2-conformal-voxelizer"   # branch with this code; set "main" once merged
    REPO = Path("/content/indoor-walk-test")
    if not REPO.exists():
        !git clone --depth 1 -b {BRANCH} https://github.com/cgm2179/indoor-walk-test.git "{REPO}"
else:
    here = Path.cwd()
    REPO = next((p for p in [here, *here.parents] if (p / "Physics Engine").is_dir()),
                Path("/Users/cameronmickle/Documents/Indoor_Walk_Test_7-7"))
SIMV2 = REPO / "Physics Engine" / "3D Map Physics" / "SIM V2"
sys.path.insert(0, str(SIMV2))
print("IN_COLAB:", IN_COLAB, "| REPO:", REPO)

IN_COLAB: False | REPO: /Users/cameronmickle/Documents/Indoor_Walk_Test_7-7


In [2]:
import numpy as np, glob, time
import jax
import bootstrap  # noqa: F401  (SIM V2 path shim)
import fw_dataset3d as D
import unet3d_train_jax as T
print("JAX backend:", jax.default_backend(), "| devices:", jax.devices())

JAX backend: cpu | devices: [CpuDevice(id=0)]


## 1 · Configuration

In [3]:
BAND      = "LTE_B71_617"    # 617 MHz
N_TX      = 6               # Tx positions (↑ to 15–30 for a real corpus; each ~1 s on GPU)
BOXES_PER = 16              # 24³ boxes cropped per Tx
NPW       = 8.0             # cells/λ for the FDTD teacher (λ/8)
REGION_M  = 7.0            # sub-volume half-extent (m)
EPOCHS    = 25             # Phase C training epochs (↑ to 100+ for real accuracy)
BASE      = 16            # U-Net base width

GRIDS = REPO / "Physics Engine" / "3D Map Physics" / "SIM V1 3D" / "conformal" / BAND
assert GRIDS.exists(), f"baked conformal grids missing: {GRIDS} (run voxelize_conformal.py --out)"
print("conformal grids:", GRIDS, "| files:", [Path(f).name for f in glob.glob(str(GRIDS/'*.npy'))])

conformal grids: /Users/cameronmickle/Documents/Indoor_Walk_Test_7-7/Physics Engine/3D Map Physics/SIM V1 3D/conformal/LTE_B71_617 | files: ['material_grid.npy', 'epsr_im.npy', 'metal_frac.npy', 'epsr_eff.npy', 'sigma_eff.npy']


## 2 · Phase B — conformal data-gen (11-channel)

Each Tx: crop the baked εr/σ/metal grids around it, zoom to λ/8, solve the continuous-medium FDTD
(`solve3d_em`), crop 24³ complex-field boxes, featurize with the 6 material one-hot **+ εr_eff + σ_eff**
(11 channels). Fixed-size crop windows → JAX compiles once.

In [4]:
t0 = time.time()
out = D.generate3d_em(BAND, n_tx=N_TX, boxes_per=BOXES_PER, npw=NPW, region_m=REGION_M,
                      grids_dir=str(GRIDS), seed=1)
shards = sorted(glob.glob(str(out / "shard_*.npz")))
d = np.load(shards[0])
print(f"Phase B: {len(shards)} shards in {time.time()-t0:.0f}s  ->  x{d['x'].shape} (11-ch)  y{d['y'].shape}")

  shard 000 boxes=16 x(11, 24, 24, 24) [conformal 11-ch]


  shard 001 boxes=16 x(11, 24, 24, 24) [conformal 11-ch]


  shard 002 boxes=16 x(11, 24, 24, 24) [conformal 11-ch]


  shard 003 boxes=16 x(11, 24, 24, 24) [conformal 11-ch]


  shard 004 boxes=16 x(11, 24, 24, 24) [conformal 11-ch]


  shard 005 boxes=16 x(11, 24, 24, 24) [conformal 11-ch]
wrote 6 conformal shards -> /Users/cameronmickle/Documents/Indoor_Walk_Test_7-7/Physics Engine/3D Map Physics/SIM V2/fw_data3d_em/LTE_B71_617
Phase B: 6 shards in 11s  ->  x(16, 11, 24, 24, 24) (11-ch)  y(16, 2, 24, 24, 24)


## 3 · Phase C — train the 11-channel pure-JAX U-Net

In [5]:
T.train(str(out), epochs=EPOCHS, base=BASE, bs=8)   # GroupNorm + optax; saves unet3d_jax.npz (cin=11)

data x(96, 11, 24, 24, 24) y(96, 2, 24, 24, 24)  cin=11 cout=2  device=cpu


  ep  0  mse=0.07800


  ep  5  mse=0.05094


  ep 10  mse=0.03986


  ep 15  mse=0.02852


  ep 20  mse=0.02251


  ep 24  mse=0.02089
saved /Users/cameronmickle/Documents/Indoor_Walk_Test_7-7/Physics Engine/3D Map Physics/SIM V2/unet3d_jax.npz


{'bk': {'c1': Array([[[[[ 4.27257568e-02, -7.52869062e-03,  5.51729929e-03],
            [ 4.64718044e-02, -2.36965604e-02, -2.07257289e-02],
            [-1.13546215e-02,  6.80124611e-02,  3.85214128e-02]],
  
           [[ 1.98595189e-02, -7.06090825e-03, -8.54675844e-03],
            [ 1.64182857e-02,  1.00326240e-02, -2.79802848e-02],
            [-4.19977913e-03,  5.85961243e-05, -1.03707723e-02]],
  
           [[-1.57443341e-02,  3.83445546e-02,  1.79556143e-02],
            [ 3.07893045e-02, -1.36202751e-02, -1.44243659e-02],
            [ 2.22828798e-02, -2.25136820e-02,  7.63168000e-03]]],
  
  
          [[[ 1.28060970e-02, -2.19964366e-02,  5.88045754e-02],
            [ 4.32920307e-02, -4.24789302e-02,  4.48024161e-02],
            [-2.03044172e-02,  4.43986617e-02, -3.52747291e-02]],
  
           [[-6.79375008e-02, -6.60177832e-03,  1.50399990e-02],
            [ 2.16206182e-02,  2.62590442e-02, -2.36507598e-02],
            [-2.97222678e-02, -1.19960671e-02, -1.47165162

## 4 · Phase D — validate vs a fresh FDTD (optional)

`T.validate(...)` gates the tiled whole-volume prediction against a fresh solve (dB RMSE / Spearman /
coherence). Accuracy improves with N_TX and EPOCHS.

In [6]:
# T.validate("unet3d_jax.npz", BAND, npw=6.0, region_m=8.0)   # uncomment after a larger run

## Notes
- **Scale**: raise `N_TX` (e.g. 15–30) and `EPOCHS` (100+) on a GPU for a real corpus; the conformal
  medium is baked once (frequency-flat εr; σ rescales per band).
- **Other bands**: `voxelize_conformal.py --band <B> --super 3 --out …/conformal/<B>` first.
- **Export**: copy the trained weights into the PyTorch `UNet3DField` (cin=11) → `torch.onnx.export`.
- vs the 9-ch model: the surrogate now sees continuous, subcell-averaged material properties — glass,
  wood, concrete, metal all distinct — instead of the hard 6-class staircase.